# Reproduction sur GPU : « profondeur 6, jeu réduit », cinq plis

Avant de lancer : **Exécution → Modifier le type d'exécution → GPU T4**, puis **Exécution → Tout exécuter**.

Le notebook exécute le code de l'équipe sans le modifier (`tools/cells/cv_config.py` et `cv_run.py`) ; seul `RUN_ONLY` change. Les résultats sont écrits dans Google Drive (`MyDrive/hmda_repro_artifacts`) : si la session est coupée, relancer tout, les plis terminés sont ignorés.

In [ ]:
# 1. Dépôt, données et Drive
import os, subprocess
from google.colab import drive
drive.mount('/content/drive')

REPO = '/content/repo'
COMMIT = '086c26b19a74fe99befdb5efd664b4968491177d'
if not os.path.isdir(REPO):
    subprocess.check_call(['git', 'clone', '-q', 'https://github.com/xiao98/demandes-de-pret-immobilier-aux-etats-unis.git', REPO])
subprocess.check_call(['git', '-C', REPO, 'checkout', '-q', COMMIT])
os.makedirs(f'{REPO}/data', exist_ok=True)
for name in ['train.csv', 'test.csv', 'sample_submission.csv']:      # les CSV sont dans l'historique du dépôt
    with open(f'{REPO}/data/{name}', 'wb') as handle:
        handle.write(subprocess.check_output(['git', '-C', REPO, 'show', f'c8b09fe:data/{name}']))
DRIVE_ARTIFACTS = '/content/drive/MyDrive/hmda_repro_artifacts'
os.makedirs(DRIVE_ARTIFACTS, exist_ok=True)
if not os.path.exists(f'{REPO}/artifacts'):
    os.symlink(DRIVE_ARTIFACTS, f'{REPO}/artifacts')
os.environ['HMDA_PROJECT_DIR'] = REPO
os.chdir(REPO)
subprocess.check_call(['python', 'tools/build_notebooks.py', '/content/gen'])
gpu = subprocess.run('nvidia-smi --query-gpu=name,memory.total --format=csv,noheader', shell=True, text=True, capture_output=True).stdout
assert gpu.strip(), "Pas de GPU : Exécution → Modifier le type d'exécution → GPU T4"
print('GPU :', gpu.strip())

In [ ]:
# 2. Notebooks 01 et 02 : données préparées, plis, variables (environ 2 min)
import nbformat

def run_notebook(name, only=None):
    nb = nbformat.read(f'/content/gen/{name}', 4)
    space = {'__name__': '__main__'}
    for index, cell in enumerate(nb.cells):
        if cell.cell_type != 'code' or (only is not None and index not in only):
            continue
        try:
            exec(compile(cell.source, f'{name}[{index}]', 'exec'), space)
        except ValueError as error:
            if 'Aucun résultat' not in str(error):      # tableaux d'ablation : résultats absents, attendu
                raise
            print(f"[{name} cellule {index}] ignorée : résultats d'ablation absents")
    return space

run_notebook('01_data_exploration.ipynb')
run_notebook('02_feature_engineering.ipynb')

In [ ]:
# 3. Cinq plis sur GPU (environ 35 min par modèle)
from pathlib import Path

G = run_notebook('04_final_model.ipynb', only={1, 2, 6, 7})      # environnement, noms, données, variables
ALL_FINAL_MODELS = False     # True : les trois modèles finaux, graine 42 (environ 2 h) au lieu du seul jeu réduit

selection = 'list(FINAL_MODEL_NAMES)' if ALL_FINAL_MODELS else '[MODEL_DEPTH6_REDUCED]'
config = Path('tools/cells/cv_config.py').read_text(encoding='utf-8')
marker = 'if RUN_ONLY is not None:'
assert config.count(marker) == 1
exec(compile(config.replace(marker, f'RUN_ONLY = {selection}\n{marker}'), 'cv_config', 'exec'), G)
G['ensure_catboost']()
G['ensure_gpu']()
print('GPU :', G['GPU_NAME'], '| groupes :', G['RUN_ONLY'])
exec(compile(Path('tools/cells/cv_run.py').read_text(encoding='utf-8'), 'cv_run', 'exec'), G)

In [ ]:
# 4. Comparaison avec les valeurs du dépôt et téléchargement
import json, shutil
import pandas as pd

out = G['OUTPUT_DIR']
runs = pd.DataFrame([json.loads(p.read_text(encoding='utf-8')) for p in sorted(out.glob('*__fold?.json'))])
table = runs.pivot(index='fold', columns='group', values='valid_auc')
reduced = G['MODEL_DEPTH6_REDUCED']
if reduced in table:
    table['dépôt'] = [0.91430, 0.92742, 0.92075, 0.91629, 0.92157]      # RESULTS.md, courbe d'apprentissage, 84 000 lignes
    table['écart'] = table[reduced] - table['dépôt']
print(table.round(5).to_string())
print('Moyennes :', table.mean().round(5).to_dict(), '| dépôt, notebook 04 : 0.92015')
print(runs[['group', 'fold', 'refit_rounds', 'seconds', 'gpu_name']].to_string(index=False))

shutil.make_archive('/content/cv_models', 'zip', out)
from google.colab import files
files.download('/content/cv_models.zip')